# `lsolve_J0` experiments
$64^3$ ($R$: $32^3$, $128^3$), $L=2\pi$, $\nu=2\times10^{-2}$, $t\le3$, $\delta t=5\times10^{-3}$, `print.in`: 9 digits.
`./plan.py list | run`; per-run parameters: `<run>/OVERRIDES`.

### Equations → code (`special/disp_current.f90`)
| equation | code |
|---|---|
| $\partial_t\mathbf A=-\mathbf E$ | `df(iax:iaz) -= p%el` |
| $\partial_t\mathbf E=\nabla\times\mathbf B-\mathbf J$ | `dEdt = c_light2*(p%curlb - mu0*p%jj_ohm)` |
| $\nabla\times\mathbf B=-\nabla^2\mathbf A+\nabla\Gamma$ | `p%curlb = -p%del2a + p%gGamma` |
| $\partial_t\Gamma=-(1-w)\,\nabla\cdot\mathbf E-w\,J^0$ | `df(iGamma)`, $w=$ `weight_longitudinalE` $=2$ |
| $\partial_t J^0=-\nabla\cdot\mathbf J$ | `df(irhoe) -= p%divJ` |
| $f^\mu=(\mathbf E\cdot\mathbf J,\;J^0\mathbf E+\mathbf J\times\mathbf B)$ | `p%ext_force(:,1:4)` → hydro/density (relativistic prefactors) |

### Definitions
| symbol | code | definition |
|---|---|---|
| $\mathbf E$ | `f(:,:,:,iex:iez)`, `p%el` | evolved |
| $J^0$ | `f(:,:,:,irhoe)`, `p%J0` | evolved; Gauss: $\nabla\cdot\mathbf E=J^0$ |
| $\Gamma$ | `f(:,:,:,iGamma)` | $\Gamma(0)=\nabla\cdot\mathbf A$ |
| $\gamma$ | `p%lorentz_gamma` | $(1-u^2)^{-1/2}$ |
| $\sigma$ | `sigma_J0` | $1/(\mu_0\eta)$, $\eta$ = `eta` (magnetic) |
| $\rho_e$ | `p%rhoe` | $J^0/\gamma-\sigma\,\mathbf u\cdot\mathbf E$ |
| $\mathbf W$ | `Wv` | $\mathbf E+\mathbf u\times\mathbf B-(\mathbf u\cdot\mathbf E)\,\mathbf u$ |
| $\mathbf J$ | `p%jj_ohm` | $J^0\mathbf u+\gamma\sigma\mathbf W$ |
| $r$ | — | $\nabla\cdot\mathbf E-J^0$ |

### $\nabla\cdot\mathbf J$ (`p%divJ`)
$$\nabla\cdot\mathbf J=\underbrace{\mathbf u\cdot\nabla J^0+J^0\,\nabla\cdot\mathbf u}_{\texttt{divJadv}}
+\underbrace{\sigma\gamma\,\nabla\cdot\mathbf W}_{\texttt{divJohm}}+\underbrace{\sigma\,\mathbf W\cdot\nabla\gamma}_{\texttt{divJgam}}$$
$$\nabla\cdot\mathbf W=J^0+\nabla\cdot(\mathbf u\times\mathbf B)-\mathbf u\cdot\nabla(\mathbf u\cdot\mathbf E)-(\mathbf u\cdot\mathbf E)\,\nabla\cdot\mathbf u,
\qquad \partial_i\gamma=\gamma^3u_j\,\partial_iu_j$$
| term | code |
|---|---|
| $\nabla J^0$ | `gJ0` = `grad(f,irhoe)` |
| $\nabla\cdot(\mathbf u\times\mathbf B)=\epsilon_{ijk}\partial_i(u_jB_k)$ | `tmp` (Levi-Civita loop with `p%uij`, `p%bij`) |
| $\partial_i(\mathbf u\cdot\mathbf E)=E_j\partial_iu_j+u_j\partial_iE_j$ | `guE`, with `Eij` = `gij(f,iex,Eij,1)`, $\texttt{Eij(:,j,i)}=\partial_iE_j$ |
| $\nabla\gamma$ | `ggam` |

$\nabla\cdot\mathbf E\to J^0$ in $\nabla\cdot\mathbf W$ $\;\Rightarrow\;\partial_t r\supset-\gamma\sigma\,r$ (`gaussdamprms`).

### Switches
| Q | switch | $\rho_e$ | $\mathbf J$ | other |
|---|---|---|---|---|
| 1 | `J0_model='relativistic'` | $J^0/\gamma-\sigma\mathbf u\cdot\mathbf E$ | $J^0\mathbf u+\gamma\sigma\mathbf W$ | reference |
| 1 | `J0_model='gamma1'` | $J^0-\sigma\mathbf u\cdot\mathbf E$ | $J^0\mathbf u+\sigma\mathbf W$ | $\gamma\to1$, $\nabla\gamma=0$ |
| 1 | `J0_model='galilean'` | $J^0$ | $J^0\mathbf u+\sigma(\mathbf E+\mathbf u\times\mathbf B)$ | $\nabla\cdot\mathbf W=J^0+\nabla\cdot(\mathbf u\times\mathbf B)$ |
| 1 | hydro `lrelativistic=F` | | | fluid: $\gamma^{-2}\to1$, $(1-c_s^2u^2)^{-1}\to1$ |
| 2 | `lJ0_no_disp=T` | $\gamma(J^0-\mathbf u\cdot\mathbf J)$ | $\nabla\times\mathbf B$ | $\mathbf E=\mathbf J/(\gamma\sigma)-\mathbf u\times\mathbf B$, $\;J^0=-\nabla\cdot(\mathbf u\times\mathbf B)-\dfrac{\mathbf J\cdot\nabla\gamma}{\gamma^2\sigma}$, $\;\partial_t\mathbf E\equiv0$, $\;\partial_t\Gamma=-J^0$ |
| 3 | `weight_longitudinalE` $=w$ | | | $\partial_t\Gamma=-(1-w)\nabla\cdot\mathbf E-wJ^0$ |
| 3 | `lJ0_divE_in_divJ=T` | | | $\nabla\cdot\mathbf W$: $J^0\to\nabla\cdot\mathbf E$ (no $-\gamma\sigma r$) |
| init | `ampl_J0pert`, `k_J0pert` | | | $J^0(0)=\nabla\cdot\mathbf E+\delta\sin(ky)$ |
| init | `initee='ohmic'`, `eta_init_ohmic` | | | $\mathbf E(0)=\eta\,\nabla\times\mathbf B/\gamma-\mathbf u\times\mathbf B$ |

### Suites
| suite | `ampluu` | $\mathbf E(0)$ | $\sigma=1/\eta$ | varied |
|---|---|---|---|---|
| A | 0.2 ($\gamma_{\max}\approx1.15$) | ohmic | 11.9 | `J0_model`, `lrelativistic`, `ampluu` $\in\{0.05,0.1,0.2\}$ |
| D | 0.2 | ohmic | 1, 3.3, 11.9, 33, 100 | `lJ0_no_disp` |
| G | 0.1 | $0.05(\cos x,\cos2z,0)$ | 11.9 (1, 100) | $w\in\{0,1,2\}$, `lJ0_divE_in_divJ`, $\delta\in\{0,10^{-3}\}$, $k=2$ |
| R | | | | $N=32,128$: `A/rel`, `A/galilean`, `D/eta8.4e-2_nodisp`, `G/w2`, `G/w0_divE`, `G/w2_pert` |

### Metric
$$\Delta_Q=\frac{|Q-Q_{\rm ref}|}{|Q_{\rm ref}|}\quad(\texttt{reldiff}),\qquad \langle\Delta_Q\rangle_t=\text{mean over }0<t\le3$$

### Numerical constraints
| case | relation | consequence |
|---|---|---|
| no $\partial_t\mathbf E$ | $\partial_t\mathbf A=\mathbf u\times\mathbf B-(\eta/\gamma)\nabla\times\nabla\times\mathbf A$ (parabolic) | $\eta\,\delta t/\delta x^2\lesssim0.1$: $\delta t=5\times10^{-4},10^{-3}$ ($\sigma=1,3.3$), $10^{-3}$ ($128^3$) |
| with $\partial_t\mathbf E$ | hyperbolic | $\delta t<\delta x/c$ |
| $\mathbf E(0)=0$ (`v1/`) | $\mathbf J(0)=\gamma\sigma\,\mathbf u\times\mathbf B$ for $\Delta t\sim1/(\gamma\sigma)$ | $\Delta u\sim uB^2$, independent of $\sigma$ → `initee='ohmic'` |

### Diagnostics (`print.in` → `data/time_series.dat`)
| name | quantity |
|---|---|
| `gaussJ0rms`, `gaussJ0max` | rms $r$, $\max\vert r\vert $ |
| `gaussdamprms` | rms $\gamma\sigma r$ |
| `grms` | rms$(\Gamma-\nabla\cdot\mathbf A)$ |
| `divJrms`, `divJadvrms`, `divJohmrms`, `divJgamrms` | rms $\nabla\cdot\mathbf J$ and its three terms |
| `Johmrms`, `J0urms`, `Jsigrms` | rms $\vert \mathbf J\vert $, $\vert J^0\mathbf u\vert $, $\vert \gamma\sigma\mathbf W\vert $ |
| `f0m`, `f0rms` | $\langle\mathbf E\cdot\mathbf J\rangle$, rms $\mathbf E\cdot\mathbf J$ |
| `fLorrms`, `fJ0Erms`, `fJxBrms` | rms $\vert J^0\mathbf E+\mathbf J\times\mathbf B\vert $, $\vert J^0\mathbf E\vert $, $\vert \mathbf J\times\mathbf B\vert $ |
| `uEm`, `uErms` | $\langle\mathbf u\cdot\mathbf E\rangle$, rms |
| `gammam`, `gammamax` | $\langle\gamma\rangle$, $\max\gamma$ |
| `J0rms`, `rhoerms`, `divErms` | rms $J^0$, $\rho_e$, $\nabla\cdot\mathbf E$ |

In [ ]:
import sys, re
from pathlib import Path
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator, LogLocator, NullLocator

sys.path.insert(0, str(Path.home() / "pencil-code" / "python"))
import pencil as pc

EXPDIR = Path.cwd() if (Path.cwd() / "plan.py").exists() else \
    Path.home() / "projects/Constrain_cleaning/runs/pencil/experiments"

mpl.rcParams.update({
    "figure.figsize": (9, 6), "figure.dpi": 110,
    "font.size": 16, "axes.titlesize": 17, "axes.labelsize": 17,
    "xtick.labelsize": 15, "ytick.labelsize": 15, "legend.fontsize": 13,
    "lines.linewidth": 2.8, "lines.markersize": 9,
    "axes.linewidth": 1.5, "axes.grid": True, "grid.alpha": 0.3,
    "xtick.major.size": 7, "ytick.major.size": 7, "xtick.major.width": 1.4, "ytick.major.width": 1.4,
    "legend.frameon": True, "legend.framealpha": 0.85,
})

def style(fig):
    '''At most 5 major ticks per axis (linear or log).'''
    for ax in fig.axes:
        for axis, scale in [(ax.xaxis, ax.get_xscale()), (ax.yaxis, ax.get_yscale())]:
            if scale == "log":
                axis.set_major_locator(LogLocator(base=10, numticks=5))
                axis.set_minor_locator(NullLocator())
            else:
                axis.set_major_locator(MaxNLocator(nbins=5))
    fig.tight_layout()

LABEL = {
    "urms": r"$u_{\rm rms}$", "brms": r"$B_{\rm rms}$", "erms": r"$E_{\rm rms}$", "jrms": r"$J_{\rm rms}$",
    "divErms": r"rms $\nabla\cdot E$", "J0rms": r"rms $J^0$", "rhoerms": r"rms $\rho_e$",
    "gaussJ0rms": r"rms$(\nabla\cdot E-J^0)$", "gaussJ0max": r"max$|\nabla\cdot E-J^0|$",
    "grms": r"rms$(\Gamma-\nabla\cdot A)$", "fJ0Erms": r"rms $|J^0E|$", "fJxBrms": r"rms $|J\times B|$",
    "f0m": r"$\langle E\cdot J\rangle$", "gammamax": r"$\max\gamma$", "Johmrms": r"rms $|J|$",
    "fLorrms": r"rms $|f_{\rm Lor}|$", "uErms": r"rms $u\cdot E$",
}
lab = lambda q: LABEL.get(q, q)

def load_ts(run):
    lines = (run / "data" / "time_series.dat").read_text().splitlines()
    header = [l for l in lines if l.startswith("#")][-1]
    names = [n for n in re.split(r"-+", header.lstrip("#")) if n]
    data = np.array([l.split() for l in lines if l.strip() and not l.startswith("#")], dtype=float)
    return {n: data[:, i] for i, n in enumerate(names)}

TS, missing = {}, []
for d in sorted(EXPDIR.glob("[ADGR]/*")):
    name = f"{d.parent.name}/{d.name}"
    if (d / "COMPLETED").exists():
        TS[name] = load_ts(d)
    else:
        missing.append(name)
print(len(TS), "finished runs;", "not finished:", missing if missing else "none")

def has(*names):
    return all(n in TS for n in names)

def reldiff(a, b):
    '''|a-b|/|b| for two runs' time series (same output times).'''
    return abs(a - b) / np.maximum(abs(b), 1e-300)

## Checks
finite; $t_{\rm end}=2.95$; $r(0)=0$ (unperturbed runs with $\partial_t\mathbf E$; in `nodisp` runs $\mathbf E$, $J^0$ are output only).

In [ ]:
for n, ts in TS.items():
    assert all(np.isfinite(v).all() for v in ts.values()), f"{n}: NaN/inf"
    assert abs(ts["t"][-1] - 2.95) < 1e-6, f"{n}: last output at t={ts['t'][-1]}"
    # no-displacement runs: E and J0 are only written for output, their Gauss residual is not meaningful
    if "pert" not in n and "nodisp" not in n:
        assert ts["gaussJ0rms"][0] == 0.0, f"{n}: Gauss residual /= 0 at t=0"
print(f"{len(TS)} runs: finite, complete to t=2.95, Gauss residual 0 at t=0 (unperturbed, with disp. current) -> OK")

## Discretization error
$\Delta_Q(64^3,128^3)$.

In [ ]:
RES_PAIRS = {"A/rel": "R/A_rel_N128", "A/galilean": "R/A_galilean_N128",
             "D/eta8.4e-2_nodisp": "R/D_eta8.4e-2_nodisp_N128", "G/w2": "R/G_w2_N128"}
QS = ["urms", "brms", "erms", "J0rms"]
RESERR = {}
fig, axes = plt.subplots(1, len(QS), figsize=(26, 6.5), sharey=True)
for ax, q in zip(axes, QS):
    for k, (a, b) in enumerate(RES_PAIRS.items()):
        if has(a, b):
            d = reldiff(TS[a][q], TS[b][q])
            RESERR[(a, q)] = d
            ax.semilogy(TS[a]["t"][1:], d[1:], color=plt.cm.tab10(k), label=a)
    ax.set_title(f"{lab(q)}: |64³ − 128³| / 128³"); ax.set_xlabel("t")
axes[0].legend()
style(fig)
for (a, q), d in RESERR.items():
    if q == "brms":
        print(f"{a:22s} brms: max relative 64^3-vs-128^3 difference {d[1:].max():.1e}")

## Q1 relativistic corrections
$\Delta_Q(\text{model},\ \texttt{A/rel})$; dashed grey: $\Delta_Q(64^3,128^3)$. $\rho_e$ is defined differently per model (compare $J^0$).

In [ ]:
MODELS = {"A/gamma1": r"$\gamma=1$ in Ohm", "A/galilean": "galilean Ohm",
          "A/rel_nrhydro": "relativistic Ohm, non-rel. fluid", "A/galilean_nrhydro": "galilean Ohm, non-rel. fluid"}
QS1 = ["urms", "brms", "erms", "J0rms", "rhoerms", "fJ0Erms"]
fig, axes = plt.subplots(2, 3, figsize=(24, 13))
for ax, q in zip(axes.flat, QS1):
    for k, (m, lbl) in enumerate(MODELS.items()):
        if has(m, "A/rel"):
            ax.semilogy(TS[m]["t"][1:], reldiff(TS[m][q], TS["A/rel"][q])[1:], color=plt.cm.tab10(k), label=lbl)
    if ("A/rel", q) in RESERR:
        ax.semilogy(TS["A/rel"]["t"][1:], RESERR[("A/rel", q)][1:], color="0.5", ls="--", lw=2, label="64³ vs 128³")
    ax.set_title(f"{lab(q)}: |model − rel| / rel"); ax.set_xlabel("t")
axes.flat[0].legend(fontsize=12)
style(fig)

$Q(t)$ for `A/*`.

In [ ]:
QS1b = ["J0rms", "rhoerms", "erms", "fJ0Erms", "gammamax", "f0m"]
runs = ["A/rel"] + list(MODELS)
fig, axes = plt.subplots(2, 3, figsize=(24, 13))
for ax, q in zip(axes.flat, QS1b):
    for k, r in enumerate(runs):
        if has(r):
            ax.plot(TS[r]["t"], TS[r][q], color=plt.cm.tab10(k) if k else "k",
                    ls="-" if k else "--", label=MODELS.get(r, "relativistic (ref)"))
    ax.set_title(lab(q)); ax.set_xlabel("t")
axes.flat[0].legend(fontsize=12)
style(fig)

### $u$ scaling
$\langle\Delta_Q\rangle_t(\text{galilean},\ \text{rel})$ vs $u_{\rm rms}=\sqrt3\,$`ampluu`; reference $\propto u^2$.

In [ ]:
U = {"0.05": ("A/u0.05_rel", "A/u0.05_galilean"), "0.1": ("A/u0.1_rel", "A/u0.1_galilean"), "0.2": ("A/rel", "A/galilean")}
fig, ax = plt.subplots(figsize=(10, 7))
print("time-averaged |galilean - rel| / rel")
for k, q in enumerate(["erms", "J0rms", "rhoerms", "brms", "urms"]):
    us, ds = [], []
    for u, (r, g) in U.items():
        if has(r, g):
            us.append(np.sqrt(3) * float(u)); ds.append(np.mean(reldiff(TS[g][q], TS[r][q])[1:]))
    if us:
        print(f"  {q:8s}", "  ".join(f"u_rms={x:.3f}: {d:.2e}" for x, d in zip(us, ds)))
        ax.loglog(us, ds, "o-", color=plt.cm.tab10(k), label=lab(q))
if us:
    uu = np.array(us)
    ax.loglog(uu, ds[-1] * (uu / uu[-1]) ** 2, color="0.3", ls="-.", lw=1.8, label=r"$\propto u^2$")
ax.set_xlabel(r"$u_{\rm rms}$ (initial)"); ax.set_ylabel("time-averaged relative difference"); ax.legend(ncol=2)
style(fig)

## Q2 displacement current
$\Delta_Q(\texttt{*\_nodisp},\ \texttt{*\_disp})$; for $\sigma=11.9$ the run with $\partial_t\mathbf E$ is `A/rel`.

In [ ]:
ETAS = {"1.": 1.0, "0.3": 0.3, "8.4e-2": 8.4e-2, "0.03": 0.03, "1e-2": 1e-2}
def disp_pair(e):
    return ("A/rel" if e == "8.4e-2" else f"D/eta{e}_disp"), f"D/eta{e}_nodisp"
QS2 = ["urms", "brms", "erms", "J0rms", "rhoerms", "f0m"]
fig, axes = plt.subplots(2, 3, figsize=(24, 13))
for ax, q in zip(axes.flat, QS2):
    for k, (e, ev) in enumerate(ETAS.items()):
        a, b = disp_pair(e)
        if has(a, b):
            ax.semilogy(TS[a]["t"][1:], reldiff(TS[b][q], TS[a][q])[1:],
                        color=plt.cm.viridis(k / (len(ETAS) - 1)), label=rf"$\sigma={1/ev:.3g}$")
    if ("A/rel", q) in RESERR:
        ax.semilogy(TS["A/rel"]["t"][1:], RESERR[("A/rel", q)][1:], color="0.5", ls="--", lw=2, label="64³ vs 128³ (σ=11.9)")
    ax.set_title(f"{lab(q)}: |no disp − disp| / disp"); ax.set_xlabel("t")
axes.flat[0].legend(fontsize=12)
style(fig)

### $\sigma$ scaling
$|\partial_t\mathbf E|/|\gamma\sigma\mathbf E|\sim\omega/\sigma\ \Rightarrow\ \langle\Delta_Q\rangle_t\propto\sigma^{-1}$.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
print("time-averaged |nodisp - disp| / disp")
for k, q in enumerate(["erms", "J0rms", "brms", "urms"]):
    sig, ds = [], []
    for e, ev in ETAS.items():
        a, b = disp_pair(e)
        if has(a, b):
            sig.append(1 / ev); ds.append(np.mean(reldiff(TS[b][q], TS[a][q])[1:]))
    if sig:
        print(f"  {q:6s}", "  ".join(f"sigma={s:6.3g}: {d:.2e}" for s, d in zip(sig, ds)))
        ax.loglog(sig, ds, "o-", color=plt.cm.tab10(k), label=lab(q))
if sig:
    ss = np.array(sig)
    ax.loglog(ss, ds[0] * (ss / ss[0]) ** -1, color="0.3", ls="-.", lw=1.8, label=r"$\propto\sigma^{-1}$")
ax.set_xlabel(r"$\sigma$"); ax.set_ylabel("time-averaged relative difference"); ax.legend()
style(fig)

$E_{\rm rms}$, $J^0_{\rm rms}$: solid with, dashed without $\partial_t\mathbf E$.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(22, 7.5))
for ax, q in zip(axes, ["erms", "J0rms"]):
    for k, (e, ev) in enumerate(ETAS.items()):
        a, b = disp_pair(e)
        c = plt.cm.viridis(k / (len(ETAS) - 1))
        if has(a, b):
            ax.semilogy(TS[a]["t"][1:], TS[a][q][1:], color=c, label=rf"$\sigma={1/ev:.3g}$")
            ax.semilogy(TS[b]["t"][1:], TS[b][q][1:], color=c, ls="--", lw=2)
    ax.set_title(f"{lab(q)}  (solid: with, dashed: without displacement current)"); ax.set_xlabel("t")
axes[0].legend()
style(fig)

### `v1/`: $\mathbf E(0)=0$
$u_{\rm rms}^{\rm nodisp}-u_{\rm rms}^{\rm disp}$: solid `v1/` ($\mathbf E(0)=0$), dashed ohmic $\mathbf E(0)$.

In [ ]:
V1 = EXPDIR / "v1"
pairs = {11.9: ("A/rel", "D/eta8.4e-2_nodisp"), 33.3: ("D/eta0.03_disp", "D/eta0.03_nodisp"),
         100.0: ("D/eta1e-2_disp", "D/eta1e-2_nodisp")}
fig, ax = plt.subplots(figsize=(11, 7))
for k, (s, (a, b)) in enumerate(pairs.items()):
    if (V1 / a / "COMPLETED").exists() and (V1 / b / "COMPLETED").exists():
        A1, B1 = load_ts(V1 / a), load_ts(V1 / b)
        ax.plot(A1["t"], B1["urms"] - A1["urms"], color=plt.cm.viridis(k / 2), label=rf"v1 ($E=0$ at $t=0$), $\sigma={s:.3g}$")
    if has(a, b):
        ax.plot(TS[a]["t"], TS[b]["urms"] - TS[a]["urms"], color=plt.cm.viridis(k / 2), ls="--", lw=2,
                label=rf"Ohmic $E$ at $t=0$, $\sigma={s:.3g}$")
ax.set_xlabel("t"); ax.set_ylabel(r"$u_{\rm rms}$(no disp.) $-$ $u_{\rm rms}$(disp.)"); ax.legend(fontsize=12)
style(fig)

## Q3 Gauss-law enforcement
$r$ and $\Gamma-\nabla\cdot\mathbf A$; top $\delta=0$, bottom $\delta=10^{-3}$.

| variant | $w$ | damping $-\gamma\sigma r$ |
|---|---|---|
| `w2` | 2 | yes |
| `w1` | 1 | yes |
| `w0` | 0 | yes |
| `w2_divE` | 2 | no |
| `w0_divE` | 0 | no |

In [ ]:
GV = {"w2": "w=2 (default)", "w1": "w=1 (Maxwell)", "w0": "w=0", "w2_divE": "w=2, no damping",
      "w0_divE": "w=0, no damping (no enforcement)"}
fig, axes = plt.subplots(2, 2, figsize=(22, 14))
for row, suffix in enumerate(["", "_pert"]):
    for ax, q in zip(axes[row], ["gaussJ0rms", "grms"]):
        for k, (v, lbl) in enumerate(GV.items()):
            r = f"G/{v}{suffix}"
            if has(r):
                y = TS[r][q]
                m = y > 0
                ax.semilogy(TS[r]["t"][m], y[m], color=plt.cm.tab10(k), label=lbl)
        ax.set_title(f"{lab(q)}" + ("  (initial violation 1e-3 sin 2y)" if suffix else "  (no initial violation)"))
        ax.set_xlabel("t")
axes[0, 0].legend(fontsize=12)
style(fig)

### Cleaning rate
$\partial_t r\supset-\gamma\sigma r$; `G/w2_pert*`; fit $r\propto e^{-\lambda t}$ for $t\le0.2$; dotted $e^{-\sigma t}$.

In [ ]:
CASES = {1.0: "G/w2_pert_eta1.", 1 / 8.4e-2: "G/w2_pert", 100.0: "G/w2_pert_eta1e-2"}
fig, ax = plt.subplots(figsize=(11, 7.5))
print("sigma   fitted decay rate   rate/sigma")
for k, (s, r) in enumerate(CASES.items()):
    if not has(r):
        continue
    t, y = TS[r]["t"], TS[r]["gaussJ0rms"]
    c = plt.cm.viridis(k / 2)
    ax.semilogy(t, y / y[0], color=c, label=rf"$\sigma={s:.3g}$")
    ax.semilogy(t, np.exp(-s * t), color=c, ls=":", lw=2)
    m = (t <= 0.2) & (y > y[0] * 1e-3)
    rate = -np.polyfit(t[m], np.log(y[m]), 1)[0] if m.sum() >= 2 else np.nan
    print(f"{s:6.3g}   {rate:12.3g}   {rate / s:8.3f}")
ax.set_ylim(1e-12, 2)
ax.set_xlabel("t"); ax.set_ylabel(r"$r_{\rm rms}(t)/r_{\rm rms}(0)$"); ax.set_title(r"dotted: $e^{-\sigma t}$")
ax.legend()
style(fig)

$\Delta_Q(\text{variant},\ \texttt{G/w2})$; dashed grey: $\Delta_Q(64^3,128^3)$.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(26, 7))
for ax, q in zip(axes, ["brms", "erms", "J0rms"]):
    for k, (v, lbl) in enumerate(GV.items()):
        r = f"G/{v}"
        if v != "w2" and has(r, "G/w2"):
            d = reldiff(TS[r][q], TS["G/w2"][q])
            ax.semilogy(TS[r]["t"][1:], np.maximum(d[1:], 1e-16), color=plt.cm.tab10(k), label=lbl)
    if ("G/w2", q) in RESERR:
        ax.semilogy(TS["G/w2"]["t"][1:], RESERR[("G/w2", q)][1:], color="0.5", ls="--", lw=2, label="64³ vs 128³")
    ax.set_title(f"{lab(q)}: |variant − w2| / w2"); ax.set_xlabel("t")
axes[0].legend(fontsize=12)
style(fig)

### $N$ dependence
$r_N(t)$, $\;p=\log_2(r_N/r_{2N})$. Perturbed: $r\approx$ remainder of $\delta$ $\Rightarrow p\approx0$.

In [ ]:
RG = {"w2": {32: "R/G_w2_N032", 64: "G/w2", 128: "R/G_w2_N128"},
      "w0_divE": {32: "R/G_w0_divE_N032", 64: "G/w0_divE", 128: "R/G_w0_divE_N128"},
      "w2_pert": {32: "R/G_w2_pert_N032", 64: "G/w2_pert", 128: "R/G_w2_pert_N128"}}
fig, axes = plt.subplots(1, 3, figsize=(26, 7), sharey=True)
for ax, (v, runs) in zip(axes, RG.items()):
    for N, r in runs.items():
        if has(r):
            y = TS[r]["gaussJ0rms"]
            ax.semilogy(TS[r]["t"][1:], y[1:], color=plt.cm.viridis({32: 0.8, 64: 0.45, 128: 0.05}[N]), label=f"N={N}")
    ax.set_title(f"{lab('gaussJ0rms')}: {GV.get(v, v + ' (perturbed)')}"); ax.set_xlabel("t")
axes[0].legend()
style(fig)
print("observed order at t=2.95, log2(r_N / r_2N):")
for v, runs in RG.items():
    r = [TS[runs[N]]["gaussJ0rms"][-1] for N in (32, 64, 128) if has(runs[N])]
    if len(r) >= 2:
        print(f"  {v:8s}", "  ".join(f"{np.log2(a / b):5.2f}" for a, b in zip(r[:-1], r[1:])))

### Residual spectra
$E_r(k,t)$ of $r_{\rm FD}=D_6\cdot\mathbf E-J^0$, `G/*_pert`.

In [ ]:
L = 2 * np.pi
def fd6_div(vx, vy, vz):
    def d(f, axis):
        h = L / f.shape[axis]
        r = lambda k: np.roll(f, -k, axis=axis)
        return (45 * (r(1) - r(-1)) - 9 * (r(2) - r(-2)) + (r(3) - r(-3))) / (60 * h)
    return d(vx, 2) + d(vy, 1) + d(vz, 0)

def shell_spectrum(field):
    fh = np.fft.fftn(field)
    nz, ny, nx = field.shape
    k1 = lambda n: np.fft.fftfreq(n, d=1.0 / n)
    KZ, KY, KX = np.meshgrid(k1(nz), k1(ny), k1(nx), indexing="ij")
    ks = np.rint(np.sqrt(KX ** 2 + KY ** 2 + KZ ** 2)).astype(int)
    E = np.bincount(ks.ravel(), weights=(abs(fh) ** 2).ravel()) / fh.size ** 2
    return np.arange(E.size), E

def residual_spectra(run):
    d = EXPDIR / run / "data"
    out = []
    for k in range(1, len(list((d / "proc0").glob("VAR[0-9]*")))):
        v = pc.read.var(datadir=str(d), ivar=k, trimall=True, quiet=True)
        kk, E = shell_spectrum(fd6_div(v.ex, v.ey, v.ez) - v.rhoe)
        out.append((float(v.t), kk, E))
    return out

fig, axes = plt.subplots(2, 3, figsize=(26, 14), sharey=True)
for ax, (v, lbl) in zip(axes.flat, GV.items()):
    r = f"G/{v}_pert"
    if not has(r):
        ax.axis("off"); continue
    spec = residual_spectra(r)
    for j, (t, kk, E) in enumerate(spec):
        m = (kk >= 1) & (E > 0)
        ax.loglog(kk[m], E[m], color=plt.cm.plasma(0.1 + 0.75 * j / max(len(spec) - 1, 1)), label=f"t={t:.2f}")
    ax.set_title(lbl); ax.set_xlabel("k"); ax.set_ylabel(r"$E_r(k)$")
axes.flat[0].legend(fontsize=12)
axes.flat[-1].axis("off")
style(fig)

## Summary
$\langle\Delta_Q\rangle_t$; last column: $\langle\Delta_B\rangle_t(64^3,128^3)$.

In [ ]:
rows = []
def row(name, a, b, ref_err_key):
    if not has(a, b):
        return
    vals = [np.mean(reldiff(TS[a][q], TS[b][q])[1:]) for q in ["urms", "brms", "erms", "J0rms"]]
    err = RESERR.get((ref_err_key, "brms"))
    rows.append((name, *vals, np.mean(err[1:]) if err is not None else np.nan))
for m, lbl in MODELS.items():
    row(f"Q1 {lbl}", m, "A/rel", "A/rel")
for e, ev in ETAS.items():
    a, b = disp_pair(e)
    row(f"Q2 no disp. current, sigma={1/ev:.3g}", b, a, "A/rel")
for v, lbl in GV.items():
    if v != "w2":
        row(f"Q3 {lbl}", f"G/{v}", "G/w2", "G/w2")
print(f"{'experiment':46s}{'urms':>10s}{'brms':>10s}{'erms':>10s}{'J0rms':>10s}{'res.err(B)':>12s}")
for r in rows:
    print(f"{r[0]:46s}" + "".join(f"{x:10.2e}" for x in r[1:5]) + f"{r[5]:12.2e}")

## Results (runs of 2026-10-21)
| | measured |
|---|---|
| discretization | $\Delta(64^3,128^3)\sim10^{-8}$ |
| Q1 galilean vs rel, $u_{\rm rms}=0.35$ | $\Delta_E=1.2\%$, $\Delta_{J^0}=1.4\%$, $\Delta_B=0.06\%$; $\;\Delta_E\propto u^{\approx3}$, $\Delta_B\propto u^{4}$ |
| Q1 `gamma1` vs rel | $\Delta_{J^0}=8.5\%$, $\Delta_B=0.8\%$ |
| Q1 `lrelativistic=F` | $\Delta_u=0.15\%$ |
| Q2 no $\partial_t\mathbf E$, $\sigma=1\to100$ | $\propto\sigma^{-1}$: $\Delta_u=4.6\times10^{-3}\to4.5\times10^{-5}$, $\Delta_E=40\%\to0.5\%$, $\Delta_{J^0}=37\%\to3.5\%$, $\Delta_B=38\%\to0.03\%$ |
| Q3 fields | $\Delta_{u,B,E}\lesssim10^{-9}$, $\Delta_{J^0}\le4\times10^{-5}$ |
| Q3 $w=0$, no damping | $r(3)=3\times10^{-7}$ ($200\times$ `w2`); $\delta\neq0$: $r(t)=r(0)$ |
| Q3 $w=2$, no damping | $r$ oscillates undamped, period $\approx1.1$ |
| Q3 with damping | $\lambda\approx\sigma$ ($1.05\sigma$ at $\sigma=100$); $w=0$: $r\to10^{-8}$ by $t\approx1$; $w=1,2$: plateau $\sim10^{-5}$ (stored in $\Gamma-\nabla\cdot\mathbf A$) |